
# Pré-processamento de Dados, do dataset de Treino, a ser usado no Desenvolvimento dos Modelos Preditivos
---

## A) Importação das Bibliotecas
---

### 1. Pacotes baixados:
---

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler, MinMaxScaler

### 2. Pacote Predexan:
---

In [2]:
import sys
import os

caminho_predexan = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if caminho_predexan not in sys.path:
    sys.path.append(caminho_predexan)

print(f"Buscando em: {caminho_predexan}")
print(f"Pastas encontradas: {os.listdir(caminho_predexan)}")

import predexan.graficos as pg
import predexan.metricas as pm
import predexan.preproc as pp

Buscando em: C:\Users\rmaca\PUC_eixo7\projeto\pmv-si-2026-1-pe7-t3-g1-pesquisaexperimentacao-2026-1\src
Pastas encontradas: ['.ipynb_checkpoints', 'analise_exploratoria_pre_processamento', 'dataset', 'imagens', 'modelos', 'predexan', 'README.md']


## B) Criação do DataFrame
---

O DataFrame, `df` é criado usando a função de `Pandas` específica para arquivos `.parquet`, chamada `pd.read_csv()`, uma vez que o dataset `UNSW_NB15_training-set.parquet` está neste formato.

In [3]:
df= pd.read_parquet('../../dataset/original/UNSW_NB15_training-set.parquet')
df.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087486,14158.942383,...,0,0,1,1,0,0,0,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473373,8395.112305,...,0,0,1,1,0,0,0,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,1572.271851,...,0,0,1,1,0,0,0,0,Normal,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,2740.178955,...,0,0,1,1,1,1,0,0,Normal,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373825,8561.499023,...,0,0,2,1,0,0,0,0,Normal,0


## C) Manipulação dos Dados
---

### 1 - Conferência dos atributos, renomeação e correção da tipagem
---

O método `.info()` mostra todos os atributos (colunas) do DataFrame, junto à quantidade de valores não nulos e tipagem de cada um, além de gasto de memória do df.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   dur                175341 non-null  float32 
 1   proto              175341 non-null  category
 2   service            175341 non-null  category
 3   state              175341 non-null  category
 4   spkts              175341 non-null  int16   
 5   dpkts              175341 non-null  int16   
 6   sbytes             175341 non-null  int32   
 7   dbytes             175341 non-null  int32   
 8   rate               175341 non-null  float32 
 9   sload              175341 non-null  float32 
 10  dload              175341 non-null  float32 
 11  sloss              175341 non-null  int16   
 12  dloss              175341 non-null  int16   
 13  sinpkt             175341 non-null  float32 
 14  dinpkt             175341 non-null  float32 
 15  sjit               175341 non-null  float32 


#### 1.1. Renomeando os atributos
---

A adequação dos nomes dos atributos é feita com o atributo `.rename()`. Para tal, há a necessidade de explicitar o parâmetro `columns` e passar como argumento os nomes antigos e novos dos atributos, em formato de `dicionário` ou `JSON`.

É feita para uma maior comodidade e compreensão no contexto de `Manipulação de Dados` e `Análise Exploratória de Dados`.

In [5]:
df = df.rename(columns={
    'dur':'duracao_conexao_seg',
    'proto':'protocolo_rede',
    'service':'servico_conexao',
    'state':'estado_conexao',
    'spkts':'num_pacotes_origem',
    'dpkts':'num_pacotes_destino',
    'sbytes':'total_bytes_origem',
    'dbytes':'total_bytes_destino',
    'rate':'taxa_pacote',
    'sload':'carga_origem',
    'dload':'carga_destino',
    'sloss':'pacotes_perdidos_origem',
    'dloss':'pacotes_perdidos_destino',
    'sinpkt':'tempo_medio_pacote_origem',
    'dinpkt':'bytes_por_pacote_destino',
    'sjit':'var_lat_pacotes_origem',
    'djit':'var_lat_pacotes_destino',
    'swin':'tamanho_janela_tcp_origem',
    'stcpb':'num_sequencia_tcp_origem',
    'dtcpb':'num_sequencia_tcp_destino',
    'dwin':'tamanho_janela_tcp_destino',
    'tcprtt':'rtt_handshake_tcp',
    'synack':'tempo_syn_synack',
    'ackdat':'tempo_synack_ack',
    'smean':'tamanho_medio_pacotes_origem',
    'dmean':'tamanho_medio_pacotes_destino',
    'trans_depth':'profundidade_transacao_http',
    'response_body_len':'tamanho_corpo_resposta_http',
    'ct_src_dport_ltm':"conexoes_recentes_ip_porta_destino",
    'ct_dst_sport_ltm':"conexoes_recentes_ip_porta_origem",
    'is_ftp_login':'ftp_autenticada',
    'ct_ftp_cmd':'num_comandos_ftp',
    'ct_flw_http_mthd':'num_metodos_http',
    'is_sm_ips_ports':'ip_porta_origem_destino_iguais',
    'attack_cat':'categoria_ataque',
    'label':'label'
})

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 36 columns):
 #   Column                              Non-Null Count   Dtype   
---  ------                              --------------   -----   
 0   duracao_conexao_seg                 175341 non-null  float32 
 1   protocolo_rede                      175341 non-null  category
 2   servico_conexao                     175341 non-null  category
 3   estado_conexao                      175341 non-null  category
 4   num_pacotes_origem                  175341 non-null  int16   
 5   num_pacotes_destino                 175341 non-null  int16   
 6   total_bytes_origem                  175341 non-null  int32   
 7   total_bytes_destino                 175341 non-null  int32   
 8   taxa_pacote                         175341 non-null  float32 
 9   carga_origem                        175341 non-null  float32 
 10  carga_destino                       175341 non-null  float32 
 11  pacotes_perdidos_origem 

#### 1.2. Conferência de Valores e Nulos
---

In [7]:
for col in df.columns:
    print(f'\nValores do atributo {col}:\n')
    print(df[col].value_counts())
    print('\n')
    print('='*40)


Valores do atributo duracao_conexao_seg:

duracao_conexao_seg
0.000009    23031
0.000003    11751
0.000008    10634
0.000005     6690
0.000004     5026
            ...  
2.013791        1
1.914309        1
3.719110        1
0.996503        1
1.557125        1
Name: count, Length: 74039, dtype: int64



Valores do atributo protocolo_rede:

protocolo_rede
tcp       79946
udp       63283
unas      12084
arp        2859
ospf       2595
          ...  
netblt       98
rdp          98
igmp         18
icmp         15
rtp           1
Name: count, Length: 133, dtype: int64



Valores do atributo servico_conexao:

servico_conexao
-           94168
dns         47294
http        18724
smtp         5058
ftp-data     3995
ftp          3428
ssh          1302
pop3         1105
dhcp           94
snmp           80
ssl            56
irc            25
radius         12
Name: count, dtype: int64



Valores do atributo estado_conexao:

estado_conexao
INT    82275
FIN    77825
CON    13152
REQ     1991
RST 

Valores nulos geram erro de processamento, por isso devem ser tratados.

No dataset escolhido, nenhuma coluna apresentou tais valores.

In [8]:
df.isna().sum()

duracao_conexao_seg                   0
protocolo_rede                        0
servico_conexao                       0
estado_conexao                        0
num_pacotes_origem                    0
num_pacotes_destino                   0
total_bytes_origem                    0
total_bytes_destino                   0
taxa_pacote                           0
carga_origem                          0
carga_destino                         0
pacotes_perdidos_origem               0
pacotes_perdidos_destino              0
tempo_medio_pacote_origem             0
bytes_por_pacote_destino              0
var_lat_pacotes_origem                0
var_lat_pacotes_destino               0
tamanho_janela_tcp_origem             0
num_sequencia_tcp_origem              0
num_sequencia_tcp_destino             0
tamanho_janela_tcp_destino            0
rtt_handshake_tcp                     0
tempo_syn_synack                      0
tempo_synack_ack                      0
tamanho_medio_pacotes_origem          0


#### 1.3. Conferência de registros duplicados em todo o DataFrame
---

Manter registros completos duplicados na base de dados que será usada para testes do modelo preditivo pode levar a `overfitting`, onde o modelo acaba "decorando" padrões específicos, que pode levar a um enviezamento de respostas. Portanto, duplicatas devem ser excluídas.

No dataset, há a presença de `78519` registros duplicados.

In [9]:
df.duplicated().sum()

np.int64(78519)

#### 1.4. Conferência de valores únicos por atributo
---

O dataset possui 11 atributos que, por terem poucos valores únicos poderiam ser divididos em `categorias`:
| Atributo | Número de valores únicos |
| :---: | :---: |
| `servico_conexao` | 13 |
| `estado_conexao` | 9 |
| `tamanho_janela_tcp_origem` | 13 |
| `tamanho_janela_tcp_destino` | 7 |
| `profundidade_transacao_http` | 11 |
| `ftp_autenticada` | 4 |
| `num_comandos_ftp` | 4 |
| `num_metodos_http` | 11 |
| `ip_porta_origem_destino_iguais` | 2 |
| `categoria_ataque` | 10 |
| `label` | 2 |

Os demais atributos possuem diversos valores únicos, não podendo ser divididos e, categorias de maneira mais simples.

In [10]:
df.nunique()

duracao_conexao_seg                   74039
protocolo_rede                          133
servico_conexao                          13
estado_conexao                            9
num_pacotes_origem                      480
num_pacotes_destino                     443
total_bytes_origem                     7214
total_bytes_destino                    6660
taxa_pacote                           76794
carga_origem                          80885
carga_destino                         77474
pacotes_perdidos_origem                 409
pacotes_perdidos_destino                370
tempo_medio_pacote_origem             76050
bytes_por_pacote_destino              74177
var_lat_pacotes_origem                77390
var_lat_pacotes_destino               76743
tamanho_janela_tcp_origem                13
num_sequencia_tcp_origem              75265
num_sequencia_tcp_destino             75089
tamanho_janela_tcp_destino                7
rtt_handshake_tcp                     43319
tempo_syn_synack                

#### 1.5. Exclusão de registros duplicados
---

Manter colunas e dados desnecessários é perigoso, pois não trazem nenhum tipo de novas informações relevantes ou características que agregam valor ao modelo preditivo, mas podem ocasionar `enviesamento` e `ruídos` que podem confundí-lo e reduzir sua eficiência e confiabilidade.

O dataset possui `175341` registros no total, mas destes, `78519` são duplicados e devem, portanto, ser apagados.  

In [11]:
df.duplicated().sum()

np.int64(78519)

In [12]:
df = df.drop_duplicates()

In [13]:
print(df.shape)

(96822, 36)


#### 1.6. Ajuste das classes dos atributos categóricos
---

Há a necessidade de se ajustar as classes raras, pois estas podem ser prejudiciais ao modelo, levando ao `overfitting`. Um cenário possível associar uma classe x, que apareceu apenas uma vez, a `0` ou `1`. 

##### 1.6.1. Atributo protocolo_rede
---

- **Original**:

In [14]:
df['protocolo_rede'].value_counts()

protocolo_rede
tcp        75805
udp        15238
ospf         769
arp          591
unas         350
           ...  
tlsp          20
xns-idp       18
igmp          15
icmp          12
rtp            1
Name: count, Length: 133, dtype: int64

- **Após limitação induzida**:

In [15]:
limite = 1000

frequencias = df['protocolo_rede'].value_counts()

protocolos_raros = frequencias[frequencias < limite].index

df['protocolo_rede'] = df['protocolo_rede'].astype('str').replace(protocolos_raros, 'outros')

print(df['protocolo_rede'].value_counts())

protocolo_rede
tcp       75805
udp       15238
outros     5779
Name: count, dtype: int64


##### 1.6.2. Atributo servico_conexao
---

- **Original**:

In [16]:
df['servico_conexao'].value_counts()

servico_conexao
-           57971
http        18090
dns          7181
smtp         5023
ftp-data     3282
ftp          2687
ssh          1295
pop3         1104
ssl            56
dhcp           51
snmp           45
irc            25
radius         12
Name: count, dtype: int64

- **Após limitação induzida**:

In [17]:
limite = 1000

frequencias = df['servico_conexao'].value_counts()

protocolos_raros = frequencias[frequencias < limite].index

df['servico_conexao'] = df['servico_conexao'].astype('str').replace(protocolos_raros, 'outros')

print(df['servico_conexao'].value_counts())

servico_conexao
-           57971
http        18090
dns          7181
smtp         5023
ftp-data     3282
ftp          2687
ssh          1295
pop3         1104
outros        189
Name: count, dtype: int64


##### 1.6.3. Atributo servico_conexao
---

- **Original**:

In [18]:
df['estado_conexao'].value_counts()

estado_conexao
FIN    74345
CON    10976
INT    10812
REQ      594
RST       83
ECO        9
PAR        1
URN        1
no         1
Name: count, dtype: int64

- **Após limitação induzida**:

In [19]:
limite = 1000

frequencias = df['estado_conexao'].value_counts()

protocolos_raros = frequencias[frequencias < limite].index

df['estado_conexao'] = df['estado_conexao'].astype('str').replace(protocolos_raros, 'outros')

print(df['estado_conexao'].value_counts())

estado_conexao
FIN       74345
CON       10976
INT       10812
outros      689
Name: count, dtype: int64


##### 1.6.4. Atributo categoria_ataque
---

- **Original**:

In [20]:
df['categoria_ataque'].value_counts()

categoria_ataque
Normal            48894
Exploits          19360
Fuzzers           14082
Reconnaissance     6000
DoS                3369
Generic            1800
Backdoor           1121
Analysis           1119
Shellcode           954
Worms               123
Name: count, dtype: int64

Não será limitado por ser um `target`. É necessário ter todos os dados detalhados desse atributo.

#### 1.7. Adequando a tipagem dos dados relativos aos atributos
---

É necessária a adequação da tipagem dos dados relativos aos atributos, para que todo o processo de `Análise Exploratória de Dados` e `Treino e Teste dos Modelos de Aprendizado Supervisionado` aconteçam da melhor maneira.

Tanto a `Matriz de Correlação de Spearman` e a biblioteca `Scikit-Learn` operam somente com valores numéricos. Em caso de haver dados das classes `str`, `object`, `datetime` e `category`, a matriz pode desconsiderá-los e o sklearn pode retornar erro.

Há a necessidade de tratar os atributos categóricos que tenham a tipagem `str` ou `object`. São muito relevantes tanto para as análises descritivas, quanto para as preditivas, por agruparem dados que possam ter grande correlação com o diagnóstico de ataque cibernético. Contudo, para serem usados, devem ser transformados em números.   

Para este notebook, que tem por finalidade o `pré-processamento dos dados` que irão ser usados no desenvolvimento dos modelos preditivos, a escolha de encoding é o `OneHotEncoding`, por ser mais adequado a modelos como `Regressão Logística` e `Multi Layer Perceptron (MLP)` por não sugerir prioridade nos dados, mas aumenta a dimensionalidade. 

##### 1.6.1. Encoding com OneHotEncoding
---

In [21]:
atr_categoricos = ['protocolo_rede', 'servico_conexao', 'estado_conexao', 'categoria_ataque', 'label']

In [22]:
encoder = OneHotEncoder()

dados_transformados = encoder.fit_transform(df[atr_categoricos]).toarray()

df_codificado = pd.DataFrame(dados_transformados, columns=encoder.get_feature_names_out(atr_categoricos), index=df.index
)


df_codificado

,protocolo_rede_outros,protocolo_rede_tcp,protocolo_rede_udp,servico_conexao_-,servico_conexao_dns,servico_conexao_ftp,servico_conexao_ftp-data,servico_conexao_http,servico_conexao_outros,servico_conexao_pop3,...,categoria_ataque_DoS,categoria_ataque_Exploits,categoria_ataque_Fuzzers,categoria_ataque_Generic,categoria_ataque_Normal,categoria_ataque_Reconnaissance,categoria_ataque_Shellcode,categoria_ataque_Worms,label_0,label_1
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175267,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
175277,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
175299,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
175305,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


##### 1.6.2. Criação do df_final para uso nos modelos
---

In [23]:
df_copia = df.copy()

df_copia = df_copia.drop(columns=atr_categoricos)

atr_nao_categoricos = df_copia.columns.to_list()

df_final = pd.concat([df_copia, df_codificado], axis=1)
df_final.info()

<class 'pandas.DataFrame'>
Index: 96822 entries, 0 to 175337
Data columns (total 59 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 96822 non-null  float32
 1   num_pacotes_origem                  96822 non-null  int16  
 2   num_pacotes_destino                 96822 non-null  int16  
 3   total_bytes_origem                  96822 non-null  int32  
 4   total_bytes_destino                 96822 non-null  int32  
 5   taxa_pacote                         96822 non-null  float32
 6   carga_origem                        96822 non-null  float32
 7   carga_destino                       96822 non-null  float32
 8   pacotes_perdidos_origem             96822 non-null  int16  
 9   pacotes_perdidos_destino            96822 non-null  int16  
 10  tempo_medio_pacote_origem           96822 non-null  float32
 11  bytes_por_pacote_destino            96822 non-null  floa

##### 1.6.3. Transformar valores em float64
---

In [24]:
for x in df_final.columns:
    df_final[x] = df_final[x].astype(float)
    print(f'O atributo {x} teve o tipo convertido para {df_final[x].dtype}')

df_final = df_final.copy()

O atributo duracao_conexao_seg teve o tipo convertido para float64
O atributo num_pacotes_origem teve o tipo convertido para float64
O atributo num_pacotes_destino teve o tipo convertido para float64
O atributo total_bytes_origem teve o tipo convertido para float64
O atributo total_bytes_destino teve o tipo convertido para float64
O atributo taxa_pacote teve o tipo convertido para float64
O atributo carga_origem teve o tipo convertido para float64
O atributo carga_destino teve o tipo convertido para float64
O atributo pacotes_perdidos_origem teve o tipo convertido para float64
O atributo pacotes_perdidos_destino teve o tipo convertido para float64
O atributo tempo_medio_pacote_origem teve o tipo convertido para float64
O atributo bytes_por_pacote_destino teve o tipo convertido para float64
O atributo var_lat_pacotes_origem teve o tipo convertido para float64
O atributo var_lat_pacotes_destino teve o tipo convertido para float64
O atributo tamanho_janela_tcp_origem teve o tipo convertid

In [25]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 96822 entries, 0 to 175337
Data columns (total 59 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 96822 non-null  float64
 1   num_pacotes_origem                  96822 non-null  float64
 2   num_pacotes_destino                 96822 non-null  float64
 3   total_bytes_origem                  96822 non-null  float64
 4   total_bytes_destino                 96822 non-null  float64
 5   taxa_pacote                         96822 non-null  float64
 6   carga_origem                        96822 non-null  float64
 7   carga_destino                       96822 non-null  float64
 8   pacotes_perdidos_origem             96822 non-null  float64
 9   pacotes_perdidos_destino            96822 non-null  float64
 10  tempo_medio_pacote_origem           96822 non-null  float64
 11  bytes_por_pacote_destino            96822 non-null  floa

### 2. Normalização de Features
---

Como visto no `Notebook de Explicação de Pré-Processamento e Análise Exploratória de Dados`, a distribuição dos dados se mostra `não normal`. Desta forma, o tipo de normalização mais adequado, de modo geral, é o `RobustScaler`, que é baseado em `IQR`. Contudo, também foi utilizado `MinMaxScaler` em alguns atributos:

1. `RobustScaler`:
   - Utilizado em atributos volumétricos: Bytes, Pacotes, Duração, Carga, RTT e Variância de Latência;
   - Presença de muitos outliers e variância muito expressiva, já que dados de volume brutos de rede são naturalmente assimétricos. Uma conexão normal pode trocar 500 bytes, enquanto um download legítimo ou um ataque de negação de serviço (DoS/DDoS) pode transferir 12 milhões de bytes na mesma janela de tempo;
   - Utiliza a Mediana e o Intervalo Interquartil (IQR) em vez da média e do máximo. Isso garante que a massa de dados normais fique bem distribuída em torno de zero, enquanto os picos (outliers) ganham valores proporcionalmente altos (como 5415.28). O modelo consegue diferenciar um tráfego comum de uma anomalia sem perder a sensibilidade dos dados menores.

2. `MinMaxScaler`:
    - Utilizado em atributos de aplicação e contagem: Métodos HTTP, Comandos FTP, Conexões Recentes, Tamanho de Janela TCP e Flags;
    - Limites bem definidos: essas variáveis possuem tetos previsíveis e controlados pelo protocolo. O número de métodos HTTP disparados em uma única sessão varia em uma escala menor. O tamanho da janela TCP possui um limite rígido do sistema (0 a 255). Não existem outliers muito grandes e distantes;
    - O MinMaxScaler mapeia com eficiência o menor valor para 0.0 e o maior valor real (seja 4, 30 ou 255) para 1.0, por conta da previsibilidade das variáveis. Isso traz todas as contagens para a mesma ordem de grandeza do restante do dataset de forma mais homogênea.

#### 2.1. Separando os atributos
---

##### 2.1.1. Atributos não categóricos
---

In [26]:
atr_nao_categoricos = df.drop(columns=atr_categoricos).columns.to_list()

##### 2.1.2. Atributos para RobustScaler
---

In [27]:
colunas_para_remover = atr_categoricos + [
    'tamanho_janela_tcp_origem', 'tamanho_janela_tcp_destino', 'profundidade_transacao_http',
    'tamanho_corpo_resposta_http', 'conexoes_recentes_ip_porta_destino', 'conexoes_recentes_ip_porta_origem',
    'ftp_autenticada', 'num_comandos_ftp', 'num_metodos_http', 'ip_porta_origem_destino_iguais'
]
atr_robust_scaler = df.drop(columns=colunas_para_remover).columns.to_list()

##### 2.1.3. Atributos para MinMaxScaler
---

In [28]:
atr_min_max_scaler = df[['tamanho_janela_tcp_origem',
    'tamanho_janela_tcp_destino',
    'profundidade_transacao_http',
    'tamanho_corpo_resposta_http',
    'conexoes_recentes_ip_porta_destino',
    'conexoes_recentes_ip_porta_origem',
    'ftp_autenticada',
    'num_comandos_ftp',
    'num_metodos_http',
    'ip_porta_origem_destino_iguais']].columns.to_list()

#### 2.2. Normalizando atributos
---

##### 2.2.1. RobustScaler
---

In [29]:
# Instancia o escalonador
escalonador_robusto = RobustScaler()

# O escalonador aprende com a sua base de dados -> necessário para salvar o aprendizado do escalonador
escalonador_robusto.fit(df_final[atr_robust_scaler])

# Transforma o bloco de colunas inteiro de uma vez para respeitar o formato esperado pelo Scikit-Learn e evitar erros de dimensionalidade no .transform()
df_robust_transformado = pd.DataFrame(
    escalonador_robusto.transform(df_final[atr_robust_scaler]), 
    columns=atr_robust_scaler, 
    index=df_final.index
)

# Iteração por todos os atributos, mostrando os valores antes e depois da Normalização e apenas transformando os dados que já foram aprendidos
for col in atr_robust_scaler:

    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df_final, col)
    print(f'\n\n   -> Atributo {col.upper()}:\n')
    print(f'O atributo {col} possui amplitude entre {valor_min} e {valor_max} e magnitude {faixa} antes da Normalização com {escalonador_robusto}.\n')
    
    
    df_final.loc[:, col] = df_robust_transformado[col]
    
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df_final, col)
    print(f'O atributo {col} foi normalizado. Sua amplitude atual está entre {valor_min} e {valor_max}. Já sua magnitude é {faixa}.\n')

# Consolida as alterações no DataFrame e previne o aviso "PerformanceWarning: DataFrame is highly fragmented"
df_final = df_final.copy()



   -> Atributo DURACAO_CONEXAO_SEG:

O atributo duracao_conexao_seg possui amplitude entre 0.0 e 59.9999885559082 e magnitude 59.9999885559082 antes da Normalização com RobustScaler().

O atributo duracao_conexao_seg foi normalizado. Sua amplitude atual está entre -0.44357456734279826 e 59.43035021560755. Já sua magnitude é 59.87392478295035.



   -> Atributo NUM_PACOTES_ORIGEM:

O atributo num_pacotes_origem possui amplitude entre 1.0 e 9616.0 e magnitude 9615.0 antes da Normalização com RobustScaler().

O atributo num_pacotes_origem foi normalizado. Sua amplitude atual está entre -0.5625 e 600.375. Já sua magnitude é 600.9375.



   -> Atributo NUM_PACOTES_DESTINO:

O atributo num_pacotes_destino possui amplitude entre 0.0 e 10974.0 e magnitude 10974.0 antes da Normalização com RobustScaler().

O atributo num_pacotes_destino foi normalizado. Sua amplitude atual está entre -0.5 e 685.375. Já sua magnitude é 685.875.



   -> Atributo TOTAL_BYTES_ORIGEM:

O atributo total_bytes_orig

##### 2.2.2. MinMaxScaler
---

In [30]:
# Instancia o escalonador
escalonador_minmax = MinMaxScaler()

# O escalonador aprende com a sua base de dados -> necessário para salvar o aprendizado do escalonador
escalonador_minmax.fit(df_final[atr_min_max_scaler])

# Transforma o bloco de colunas inteiro de uma vez para respeitar o formato esperado pelo Scikit-Learn e evitar erros de dimensionalidade no .transform()
df_minmax_transformado = pd.DataFrame(
    escalonador_minmax.transform(df_final[atr_min_max_scaler]), 
    columns=atr_min_max_scaler, 
    index=df_final.index
)

# Iteração por todos os atributos, mostrando os valores antes e depois da Normalização e apenas transformando os dados que já foram aprendidos
for col in atr_min_max_scaler:

    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df_final, col)
    print(f'\n\n   -> Atributo {col.upper()}:\n')
    print(f'O atributo {col} possui amplitude entre {valor_min} e {valor_max} e magnitude {faixa} antes da Normalização com {escalonador_minmax}.\n')
    
    df_final.loc[:, col] = df_minmax_transformado[col]
    
    _, valor_min, valor_max, _, _, faixa = pm.gerar_dispersao(df_final, col)
    print(f'O atributo {col} foi normalizado. Sua amplitude atual está entre {valor_min} e {valor_max}. Já sua magnitude é {faixa}.\n')

# Consolida as alterações no DataFrame e previne o aviso "PerformanceWarning: DataFrame is highly fragmented"
df_final = df_final.copy()



   -> Atributo TAMANHO_JANELA_TCP_ORIGEM:

O atributo tamanho_janela_tcp_origem possui amplitude entre 0.0 e 255.0 e magnitude 255.0 antes da Normalização com MinMaxScaler().

O atributo tamanho_janela_tcp_origem foi normalizado. Sua amplitude atual está entre 0.0 e 1.0. Já sua magnitude é 1.0.



   -> Atributo TAMANHO_JANELA_TCP_DESTINO:

O atributo tamanho_janela_tcp_destino possui amplitude entre 0.0 e 255.0 e magnitude 255.0 antes da Normalização com MinMaxScaler().

O atributo tamanho_janela_tcp_destino foi normalizado. Sua amplitude atual está entre 0.0 e 1.0. Já sua magnitude é 1.0.



   -> Atributo PROFUNDIDADE_TRANSACAO_HTTP:

O atributo profundidade_transacao_http possui amplitude entre 0.0 e 172.0 e magnitude 172.0 antes da Normalização com MinMaxScaler().

O atributo profundidade_transacao_http foi normalizado. Sua amplitude atual está entre 0.0 e 1.0. Já sua magnitude é 1.0.



   -> Atributo TAMANHO_CORPO_RESPOSTA_HTTP:

O atributo tamanho_corpo_resposta_http possui a

##### 2.2.3. Salvando os escalonadores para serem usados com o dataset de teste
---

- **RobustScaler**:

In [31]:
joblib.dump(escalonador_robusto, '../escalonadores/escalonador_robusto.pkl')

['../escalonadores/escalonador_robusto.pkl']

- **MinMaxScaler**:

In [32]:
joblib.dump(escalonador_minmax, '../escalonadores/escalonador_minmax.pkl')

['../escalonadores/escalonador_minmax.pkl']

## E) Criação do Dataset Pré-processado de Treino
---

Este dataset será usado treino dos modelos preditivos.

In [33]:
df.to_parquet('../../dataset/pre_processado/treino_pre_processado.parquet')